# Experiment 3: SRAM+DRAM Caching Policy Comparisons

**Baseline**: `M3-Baseline.ipynb` running `full_EdgeRAG_v3.yaml` on `arch/tpu_v4i.yaml`  
(gte-base-en-v1.5 encoder + flat dense retrieval + Sheared-LLaMA-2.7B decoder, INT8)

> **Scope**: We use TPU v4i-inspired tiers with `GlobalBuffer` in the access path,
> `MainMemory` as DRAM cache, and `Disk` as backing store. Experiment 3 compares
> `lfu_l1`, `lfu_l2`, `lfu_l1_l2`, and `opt_two_tier` against `no_cache`.
> Latency results are reported as **TTFT** contributions.

## Hypotheses
1. Memory accesses dominate energy and TTFT at large retrieval scales.
2. DRAM caching reduces TTFT at the cost of higher DRAM traffic energy.
3. There exists a break-even frontier where caching transitions from costing more energy
   than the no-cache baseline to costing less, as DRAM capacity and reuse ratio grow.

## What this notebook does
- Loads the AccelForge baseline from `baseline_cache.pkl` (produced by the last cell of `M3-Baseline.ipynb`).
- Uses hardware constants from `arch/tpu_v4i.yaml`: GlobalBuffer (treated as DRAM cache) at 2048 GB/s read / 1024 GB/s write, MainMemory (treated as Disk-tier backing store) at 614 GB/s.
- Generates a synthetic Zipf-distributed document-access trace per BEIR reuse ratio (Table 2).
- Runs the triple sweep `(DRAM_GB, reuse_ratio, policy)` through `cache_sim.py`.
- Converts hit/miss counts to energy, TTFT latency, and EDP, adding the non-SIM AccelForge baseline.
- Runs a full sanity-check suite before producing plots.
- Produces heatmaps, the LFU-vs-OPT gap plot, and a hit-rate bar chart.

**Results are reported as-is. No interpretation of whether they support or contradict the hypotheses is made here.**

---
### Note on parameter scale
The AccelForge baseline (`M3-Baseline.ipynb`) maps the EdgeRAG workload at
`N_DOCS = 5 000 000`, `K = 10`, `N_TOKENS = 512`. Per-query full-pipeline and SIM-term
(`doc_scores`) costs depend on the workload + arch combination; current run uses
`full_EdgeRAG_v3.yaml` on `arch/tpu_v4i.yaml` and the actual values are reported by Cell 6.

The cache simulator is intentionally decoupled from AccelForge `N_DOCS`: it only needs a
corpus large enough that the DRAM sweep crosses the eviction frontier. We use
`N_DOCS_TRACE = 2 000 000`, `N_QUERIES = 100 000`, `K_SIM = 10`, and a DRAM sweep of
`{0.25, 0.5, 1, 2, 4} GB`. With reuse ratios from BEIR Table 2, the worst-case unique-doc
count is ~800 k, which forces evictions at 0.25 GB / 0.5 GB and saturates the cache at 1+ GB.
Override `N_DOCS_TRACE` and `N_QUERIES` in Cell 2 and re-run if you want to shift the frontier.

## Cell 1 — Imports

In [1]:
import sys, os, pickle
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

sys.path.insert(0, '/home/workspace/workspace')
sys.path.insert(0, '/home/workspace/workspace/scripts')

from cache_sim import (
    synth_trace,
    simulate_no_cache,
    simulate_lfu_cost_aware,
    simulate_two_tier_lfu,
    simulate_two_tier_opt,
    simulate_belady_opt,
    CacheResult,
)

%matplotlib inline
print('Imports OK')

Imports OK


## Cell 2 — Parameters

In [2]:
# ── AccelForge workload parameters (must match M3-Baseline) ──────────────────
N_DOCS    = 5_000_000
N_TOKENS  = 512
K         = 10
L         = 512

# ── Cache-simulation parameters ───────────────────────────────────────────────
N_DOCS_TRACE = 2_000_000
N_QUERIES    = 100_000
K_SIM        = 10
DECAY_FACTOR = 0.99
SEED         = 42

# ── Sweeps ────────────────────────────────────────────────────────────────────
# GlobalBuffer (SRAM) is fixed at TPU v4i default; MainMemory (DRAM) is swept.
GLOBAL_BUFFER_MB = 128
MAIN_MEMORY_GB_VALUES = [1, 2, 4, 8, 16]

# Backward-compat alias so downstream plotting cells still work.
DRAM_GB_VALUES = MAIN_MEMORY_GB_VALUES

# ── Reuse ratios from BEIR Table 2 (EdgeRAG) ─────────────────────────────────
BEIR_DATASETS = [
    ('nq',       1.25),
    ('hotpotqa', 1.42),
    ('scidocs',  1.73),
    ('quora',    1.91),
    ('fever',    2.41),
    ('fiqa',     4.47),
]
DATASET_NAMES = [d for d, _ in BEIR_DATASETS]
REUSE_RATIOS  = [r for _, r in BEIR_DATASETS]

# ── Hardware constants from tpu_v4i.yaml defaults ────────────────────────────
# Cache hierarchy modelled in Exp2:
#   GlobalBuffer (SRAM cache) -> MainMemory (DRAM cache) -> Disk (NVMe backing)
# LocalBuffer is intentionally excluded from cache policy in this experiment.
ENC_EMBED_DIM = 768
BITS_PER_VAL  = 8
EMB_BITS      = ENC_EMBED_DIM * BITS_PER_VAL
EMB_BYTES     = EMB_BITS // 8

# GlobalBuffer (SRAM) read cost.
L2_READ_ENERGY_pJ_per_bit = 1.88
L2_READ_BW_GBps           = 2048.0

# MainMemory (DRAM) read cost.
DRAM_READ_ENERGY_pJ_per_bit = 7.03
DRAM_READ_BW_GBps           = 614.0

# Disk (NVMe random read) constants from user methodology.
DISK_READ_ENERGY_J_per_access = 10e-9
DISK_READ_LATENCY_s_per_access = 100e-6
DISK_BW_GBps = 7.0  # label-only convenience

# Aliases kept for existing labels/captions in later plotting cells.
DRAM_ENERGY_pJ_per_bit = DRAM_READ_ENERGY_pJ_per_bit
DRAM_BW_GBps = DRAM_READ_BW_GBps

# ── Capacity conversion ───────────────────────────────────────────────────────
def embedding_capacity_gb(size_gb: float, n_docs_trace: int) -> int:
    slots = int(size_gb * 1024**3) // EMB_BYTES
    return min(slots, n_docs_trace)

def embedding_capacity_mb(size_mb: float, n_docs_trace: int) -> int:
    slots = int(size_mb * 1024**2) // EMB_BYTES
    return min(slots, n_docs_trace)

# Backward-compat alias expected by legacy cells.
def embedding_capacity(dram_gb: float, n_docs_trace: int) -> int:
    return embedding_capacity_gb(dram_gb, n_docs_trace)

GLOBAL_BUFFER_CAPACITY = embedding_capacity_mb(GLOBAL_BUFFER_MB, N_DOCS_TRACE)

# ── Per-access costs (pJ / ns) using locked equation ─────────────────────────
# e_per_access = h2*E_L2 + h_dram*(E_L2 + E_DRAM) + h_disk*(E_L2 + E_DRAM + E_DISK)
E_L2_pJ = L2_READ_ENERGY_pJ_per_bit * EMB_BITS
E_DRAM_pJ = DRAM_READ_ENERGY_pJ_per_bit * EMB_BITS
E_DISK_pJ = DISK_READ_ENERGY_J_per_access * 1e12

TTFT_L2_ns = EMB_BITS / (L2_READ_BW_GBps * 1e9 * 8) * 1e9
TTFT_DRAM_ns = EMB_BITS / (DRAM_READ_BW_GBps * 1e9 * 8) * 1e9
TTFT_DISK_ns = DISK_READ_LATENCY_s_per_access * 1e9

# Legacy aliases used by older cells in this notebook.
E_HIT_pJ = E_L2_pJ
E_MISS_pJ = E_L2_pJ + E_DRAM_pJ + E_DISK_pJ
TTFT_HIT_ns = TTFT_L2_ns
TTFT_MISS_ns = TTFT_L2_ns + TTFT_DRAM_ns + TTFT_DISK_ns

print(f'Embedding row: {EMB_BYTES} bytes ({BITS_PER_VAL}-bit INT)')
print(f'GlobalBuffer fixed at {GLOBAL_BUFFER_MB} MB -> {GLOBAL_BUFFER_CAPACITY:,} embeddings')
print(f'Per-access constants: E_L2={E_L2_pJ:.1f} pJ, E_DRAM={E_DRAM_pJ:.1f} pJ, E_DISK={E_DISK_pJ:.1f} pJ')
print(f'Per-access TTFT: L2={TTFT_L2_ns:.3f} ns, DRAM={TTFT_DRAM_ns:.3f} ns, DISK={TTFT_DISK_ns:.1f} ns')
print()
for gb in MAIN_MEMORY_GB_VALUES:
    cap = embedding_capacity_gb(gb, N_DOCS_TRACE)
    pct = min(100.0, 100.0 * cap / N_DOCS_TRACE)
    print(f'  MainMemory {gb:>2} GB -> {cap:>10,} embeddings ({pct:.1f}% of trace corpus)')

Embedding row: 768 bytes (8-bit INT)
GlobalBuffer fixed at 128 MB -> 174,762 embeddings
Per-access constants: E_L2=11550.7 pJ, E_DRAM=43192.3 pJ, E_DISK=10000.0 pJ
Per-access TTFT: L2=0.375 ns, DRAM=1.251 ns, DISK=100000.0 ns

  MainMemory  1 GB ->  1,398,101 embeddings (69.9% of trace corpus)
  MainMemory  2 GB ->  2,000,000 embeddings (100.0% of trace corpus)
  MainMemory  4 GB ->  2,000,000 embeddings (100.0% of trace corpus)
  MainMemory  8 GB ->  2,000,000 embeddings (100.0% of trace corpus)
  MainMemory 16 GB ->  2,000,000 embeddings (100.0% of trace corpus)


## Cell 3 — Load AccelForge baseline

In [3]:
BASELINE_PATH = '/home/workspace/workspace/baseline_cache.pkl'

with open(BASELINE_PATH, 'rb') as f:
    bl = pickle.load(f)

baseline_energy_J  = bl['baseline_energy_J']   # einsum_name -> J
baseline_latency_s = bl['baseline_latency_s']  # einsum_name -> s
SIM_EINSUM         = bl['sim_einsum']          # 'doc_scores'

# Non-SIM totals: everything except the similarity/retrieval einsum.
# We replace that einsum's memory traffic with the simulator output.
non_sim_einsums = [e for e in baseline_energy_J if e != SIM_EINSUM]

E_nonsim_pJ = sum(baseline_energy_J[e] for e in non_sim_einsums) * 1e12
L_nonsim_ns = sum(baseline_latency_s[e] for e in non_sim_einsums) * 1e9

print(f'Loaded baseline from {BASELINE_PATH}')
print(f'SIM einsum: "{SIM_EINSUM}"')
print(f'Non-SIM einsums ({len(non_sim_einsums)}): {non_sim_einsums[:5]} ...')
print(f'Non-SIM total energy: {E_nonsim_pJ:.0f} pJ')
print(f'Non-SIM total latency: {L_nonsim_ns:.0f} ns')

Loaded baseline from /home/workspace/workspace/baseline_cache.pkl
SIM einsum: "FaissScoreComputation"
Non-SIM einsums (26): ['EncInputLoad', 'EncQueryProj', 'EncKeyProj', 'EncValueProj', 'EncAttentionScores'] ...
Non-SIM total energy: 202999508880 pJ
Non-SIM total latency: 20434747 ns


## Cell 4 — Synthesize access traces (one per BEIR reuse ratio)

In [4]:
print('Generating Zipf traces...')
traces      = {}
trace_stats = {}
for name, reuse in BEIR_DATASETS:
    t = synth_trace(
        n_queries=N_QUERIES,
        n_docs=N_DOCS_TRACE,
        reuse_ratio=reuse,
        k_per_query=K_SIM,
        seed=SEED,
    )
    unique       = len(set(t))
    actual_reuse = len(t) / unique if unique > 0 else float('inf')
    traces[name] = t
    trace_stats[name] = {
        'target_reuse': reuse,
        'actual_reuse': actual_reuse,
        'unique':       unique,
        'total':        len(t),
    }
    print(f'  {name:<10} reuse target={reuse:.2f}  '
          f'actual={actual_reuse:.2f}  '
          f'unique={unique:,}  total={len(t):,}')

print('Done.')

Generating Zipf traces...


  nq         reuse target=1.25  actual=1.27  unique=787,061  total=1,000,000


  hotpotqa   reuse target=1.42  actual=1.42  unique=704,414  total=1,000,000


  scidocs    reuse target=1.73  actual=1.73  unique=578,185  total=1,000,000


  quora      reuse target=1.91  actual=1.91  unique=523,958  total=1,000,000


  fever      reuse target=2.41  actual=2.40  unique=416,178  total=1,000,000


  fiqa       reuse target=4.47  actual=4.33  unique=231,047  total=1,000,000
Done.


## Cell 5 — Run the triple sweep (DRAM_GB × reuse_ratio × policy)

In [5]:
POLICIES = ['no_cache', 'lfu_l1', 'lfu_l2', 'lfu_l1_l2', 'opt_two_tier']

sim_results = {gb: {name: {} for name, _ in BEIR_DATASETS} for gb in DRAM_GB_VALUES}
tier_counts = {gb: {name: {} for name, _ in BEIR_DATASETS} for gb in DRAM_GB_VALUES}

ttft_cost_uniform = {}

total = len(DRAM_GB_VALUES) * len(BEIR_DATASETS)
done = 0

for gb in DRAM_GB_VALUES:
    mainmem_capacity = embedding_capacity_gb(gb, N_DOCS_TRACE)
    for name, _reuse in BEIR_DATASETS:
        trace = traces[name]

        no_cache = simulate_no_cache(trace)
        lfu_l1 = simulate_lfu_cost_aware(trace, GLOBAL_BUFFER_CAPACITY, ttft_cost_uniform, decay_factor=DECAY_FACTOR)
        lfu_l2 = simulate_lfu_cost_aware(trace, mainmem_capacity, ttft_cost_uniform, decay_factor=DECAY_FACTOR)
        lfu_l1_l2 = simulate_two_tier_lfu(
            trace,
            l1_capacity=GLOBAL_BUFFER_CAPACITY,
            l2_capacity=mainmem_capacity,
            gen_latency=ttft_cost_uniform,
            decay_factor=DECAY_FACTOR,
            check_invariants=False,
        )
        opt_two_tier = simulate_two_tier_opt(
            trace,
            l1_capacity=GLOBAL_BUFFER_CAPACITY,
            l2_capacity=mainmem_capacity,
            check_invariants=False,
        )

        sim_results[gb][name]['no_cache'] = no_cache
        tier_counts[gb][name]['no_cache'] = {'h2': 0, 'h_dram': 0, 'h_disk': no_cache.misses}

        sim_results[gb][name]['lfu_l1'] = lfu_l1
        tier_counts[gb][name]['lfu_l1'] = {'h2': lfu_l1.hits, 'h_dram': 0, 'h_disk': lfu_l1.misses}

        sim_results[gb][name]['lfu_l2'] = lfu_l2
        tier_counts[gb][name]['lfu_l2'] = {'h2': 0, 'h_dram': lfu_l2.hits, 'h_disk': lfu_l2.misses}

        sim_results[gb][name]['lfu_l1_l2'] = CacheResult(
            hits=lfu_l1_l2.l1_hits + lfu_l1_l2.l2_hits,
            misses=lfu_l1_l2.misses,
            hit_rate=lfu_l1_l2.overall_hit_rate,
            miss_positions=lfu_l1_l2.miss_positions,
        )
        tier_counts[gb][name]['lfu_l1_l2'] = {'h2': lfu_l1_l2.l1_hits, 'h_dram': lfu_l1_l2.l2_hits, 'h_disk': lfu_l1_l2.misses}

        sim_results[gb][name]['opt_two_tier'] = CacheResult(
            hits=opt_two_tier.l1_hits + opt_two_tier.l2_hits,
            misses=opt_two_tier.misses,
            hit_rate=opt_two_tier.overall_hit_rate,
            miss_positions=opt_two_tier.miss_positions,
        )
        tier_counts[gb][name]['opt_two_tier'] = {'h2': opt_two_tier.l1_hits, 'h_dram': opt_two_tier.l2_hits, 'h_disk': opt_two_tier.misses}

        done += 1
        print(f'  [{done}/{total}] MainMemory={gb} GB  dataset={name}')

print('Sweep complete.')

  [1/30] MainMemory=1 GB  dataset=nq


  [2/30] MainMemory=1 GB  dataset=hotpotqa


  [3/30] MainMemory=1 GB  dataset=scidocs


  [4/30] MainMemory=1 GB  dataset=quora


  [5/30] MainMemory=1 GB  dataset=fever


...

  [26/30] MainMemory=16 GB  dataset=hotpotqa


  [27/30] MainMemory=16 GB  dataset=scidocs


  [28/30] MainMemory=16 GB  dataset=quora


  [29/30] MainMemory=16 GB  dataset=fever


  [30/30] MainMemory=16 GB  dataset=fiqa
Sweep complete.


## Cell 6 — Convert hit/miss counts to energy, latency, and EDP per query

In [6]:
# Per-query totals (SIM/doc_scores AccelForge term replaced by cache-modelled retrieval):
#   energy (pJ)    = non-SIM baseline + hits × E_HIT   + misses × E_MISS
#   ttft_ns        = non-SIM baseline + hits × TTFT_HIT + misses × TTFT_MISS
#   edp            = energy × ttft_ns
#
# We also expose "cache-only" components that strip the non-SIM baseline so
# heatmaps can show the cache's actual contribution.  The non-SIM baseline is
# decoder-dominated (~5.5 J/query) and dwarfs cache traffic (~1 µJ/query),
# which would otherwise wash out all per-policy / per-DRAM differences.

def per_query_metrics(result: CacheResult, n_queries: int):
    h = result.hits   / n_queries
    m = result.misses / n_queries
    cache_e_pJ    = h * E_HIT_pJ    + m * E_MISS_pJ
    cache_ttft_ns = h * TTFT_HIT_ns + m * TTFT_MISS_ns
    e_pJ    = E_nonsim_pJ + cache_e_pJ
    ttft_ns = L_nonsim_ns + cache_ttft_ns
    edp     = e_pJ * ttft_ns
    return {
        'energy_pJ':       e_pJ,
        'ttft_ns':         ttft_ns,
        'edp':             edp,
        'cache_energy_pJ': cache_e_pJ,
        'cache_ttft_ns':   cache_ttft_ns,
    }

ndram = len(DRAM_GB_VALUES)
ndata = len(BEIR_DATASETS)

energy        = {p: np.zeros((ndram, ndata)) for p in POLICIES}
ttft_latency  = {p: np.zeros((ndram, ndata)) for p in POLICIES}
edp           = {p: np.zeros((ndram, ndata)) for p in POLICIES}
cache_energy  = {p: np.zeros((ndram, ndata)) for p in POLICIES}
cache_ttft    = {p: np.zeros((ndram, ndata)) for p in POLICIES}
hitrate       = {p: np.zeros((ndram, ndata)) for p in POLICIES}

for i, gb in enumerate(DRAM_GB_VALUES):
    for j, (name, _) in enumerate(BEIR_DATASETS):
        for p in POLICIES:
            res = sim_results[gb][name][p]
            m   = per_query_metrics(res, N_QUERIES)
            energy[p][i, j]        = m['energy_pJ']
            ttft_latency[p][i, j]  = m['ttft_ns']
            edp[p][i, j]           = m['edp']
            cache_energy[p][i, j]  = m['cache_energy_pJ']
            cache_ttft[p][i, j]    = m['cache_ttft_ns']
            hitrate[p][i, j]       = res.hit_rate

# Original SIM term that the cache replaces (for context in the figure caption)
sim_energy_J  = baseline_energy_J[SIM_EINSUM]
sim_latency_s = baseline_latency_s[SIM_EINSUM]
nonsim_energy_J  = E_nonsim_pJ * 1e-12
nonsim_latency_s = L_nonsim_ns * 1e-9

print('Built energy / TTFT / EDP / cache-only matrices.')
print()
print(f'AccelForge full baseline (with SIM):     '
      f'E={(nonsim_energy_J + sim_energy_J)*1e3:>9.1f} mJ,  '
      f'TTFT={(nonsim_latency_s + sim_latency_s)*1e3:>9.1f} ms / query')
print(f'AccelForge non-SIM baseline (cache off): '
      f'E={nonsim_energy_J*1e3:>9.1f} mJ,  '
      f'TTFT={nonsim_latency_s*1e3:>9.1f} ms / query')
print(f'SIM term replaced by cache (doc_scores): '
      f'E={sim_energy_J*1e3:>9.1f} mJ,  '
      f'TTFT={sim_latency_s*1e3:>9.1f} ms / query')
print()
print(f'Cache traffic at no_cache (all miss):    '
      f'E={cache_energy["no_cache"][0,0]*1e-3:>9.3f} nJ, '
      f'TTFT={cache_ttft["no_cache"][0,0]*1e-3:>9.3f} µs / query')
print(f'  → cache replaces {sim_energy_J*1e3:.1f} mJ / {sim_latency_s:.2f} s of SIM compute with cache-modelled traffic')

Built energy / TTFT / EDP / cache-only matrices.

AccelForge full baseline (with SIM):     E=    289.9 mJ,  TTFT=     22.9 ms / query
AccelForge non-SIM baseline (cache off): E=    203.0 mJ,  TTFT=     20.4 ms / query
SIM term replaced by cache (doc_scores): E=     86.9 mJ,  TTFT=      2.5 ms / query

Cache traffic at no_cache (all miss):    E=  647.430 nJ, TTFT= 1000.016 µs / query
  → cache replaces 86.9 mJ / 0.00 s of SIM compute with cache-modelled traffic


In [7]:
# Override cache-only metrics using the locked h2/h_dram/h_disk model.
# This supersedes the legacy hit/miss-only conversion cell above.

def per_query_metrics_hierarchy(h2_count: int, h_dram_count: int, h_disk_count: int, n_queries: int):
    h2 = h2_count / n_queries
    h_dram = h_dram_count / n_queries
    h_disk = h_disk_count / n_queries

    cache_e_pJ = (
        h2 * E_L2_pJ
        + h_dram * (E_L2_pJ + E_DRAM_pJ)
        + h_disk * (E_L2_pJ + E_DRAM_pJ + E_DISK_pJ)
    )
    cache_ttft_ns = (
        h2 * TTFT_L2_ns
        + h_dram * (TTFT_L2_ns + TTFT_DRAM_ns)
        + h_disk * (TTFT_L2_ns + TTFT_DRAM_ns + TTFT_DISK_ns)
    )

    e_pJ = E_nonsim_pJ + cache_e_pJ
    ttft_ns = L_nonsim_ns + cache_ttft_ns
    edp = e_pJ * ttft_ns
    return {
        'energy_pJ': e_pJ,
        'ttft_ns': ttft_ns,
        'edp': edp,
        'cache_energy_pJ': cache_e_pJ,
        'cache_ttft_ns': cache_ttft_ns,
    }

ndram = len(DRAM_GB_VALUES)
ndata = len(BEIR_DATASETS)

energy = {p: np.zeros((ndram, ndata)) for p in POLICIES}
ttft_latency = {p: np.zeros((ndram, ndata)) for p in POLICIES}
edp = {p: np.zeros((ndram, ndata)) for p in POLICIES}
cache_energy = {p: np.zeros((ndram, ndata)) for p in POLICIES}
cache_ttft = {p: np.zeros((ndram, ndata)) for p in POLICIES}
hitrate = {p: np.zeros((ndram, ndata)) for p in POLICIES}

for i, gb in enumerate(DRAM_GB_VALUES):
    for j, (name, _) in enumerate(BEIR_DATASETS):
        for p in POLICIES:
            c = tier_counts[gb][name][p]
            res = sim_results[gb][name][p]
            m = per_query_metrics_hierarchy(c['h2'], c['h_dram'], c['h_disk'], N_QUERIES)
            energy[p][i, j] = m['energy_pJ']
            ttft_latency[p][i, j] = m['ttft_ns']
            edp[p][i, j] = m['edp']
            cache_energy[p][i, j] = m['cache_energy_pJ']
            cache_ttft[p][i, j] = m['cache_ttft_ns']
            hitrate[p][i, j] = res.hit_rate

print('Hierarchy override complete: energy/TTFT now use h2/h_dram/h_disk.')

Hierarchy override complete: energy/TTFT now use h2/h_dram/h_disk.


In [8]:
## Cell 7 — Sanity Checks
# Run before producing any plots. Every FAIL must be investigated.

def check(cond, msg):
    status = 'PASS' if cond else 'FAIL'
    print(f'[{status}] {msg}')
    return bool(cond)

all_ok         = True
total_accesses = N_QUERIES * K_SIM

# ── Trace checks ─────────────────────────────────────────────────────────────
for name, _reuse in BEIR_DATASETS:
    s = trace_stats[name]
    all_ok &= check(
        abs(s['actual_reuse'] - s['target_reuse']) / s['target_reuse'] < 0.05,
        f'{name}: actual reuse within 5% of target',
    )
    all_ok &= check(s['total'] == total_accesses, f'{name}: trace length = N_QUERIES × K_SIM')

# ── Capacity checks ───────────────────────────────────────────────────────────
for gb in DRAM_GB_VALUES:
    cap = embedding_capacity(gb, N_DOCS_TRACE)
    all_ok &= check(cap <= N_DOCS_TRACE, f'{gb} GB: capacity capped at N_DOCS_TRACE')

# ── Policy ordering checks ────────────────────────────────────────────────────
example_keys = sim_results[DRAM_GB_VALUES[0]][BEIR_DATASETS[0][0]].keys()
lfu_key = 'lfu_l1_l2' if 'lfu_l1_l2' in example_keys else ('lfu' if 'lfu' in example_keys else 'dram_lfu')
opt_key = 'opt_two_tier' if 'opt_two_tier' in example_keys else 'opt'

for gb in DRAM_GB_VALUES:
    for name, _reuse in BEIR_DATASETS:
        nc = sim_results[gb][name]['no_cache']
        lf = sim_results[gb][name][lfu_key]
        op = sim_results[gb][name][opt_key]
        all_ok &= check(nc.hits == 0,                              f'{gb} GB {name}: no-cache hits = 0')
        all_ok &= check(nc.misses == total_accesses,               f'{gb} GB {name}: no-cache misses = total')
        all_ok &= check(lf.hits + lf.misses == total_accesses,     f'{gb} GB {name}: {lfu_key} total valid')
        all_ok &= check(op.hits + op.misses == total_accesses,     f'{gb} GB {name}: {opt_key} total valid')
        all_ok &= check(op.misses <= lf.misses <= nc.misses,       f'{gb} GB {name}: {opt_key} <= {lfu_key} <= no-cache')

# ── Latency and energy ordering ───────────────────────────────────────────────
all_ok &= check(TTFT_MISS_ns > TTFT_HIT_ns, 'miss TTFT > hit TTFT')
all_ok &= check(E_MISS_pJ   >= E_HIT_pJ,   'miss energy >= hit energy')

# ── OPT monotonicity ─────────────────────────────────────────────────────────
for j, (name, _reuse) in enumerate(BEIR_DATASETS):
    opt_rates = [hitrate[opt_key][i, j] for i in range(ndram)]
    all_ok &= check(
        all(opt_rates[i] <= opt_rates[i + 1] + 1e-12 for i in range(len(opt_rates) - 1)),
        f'{name}: OPT hit rate non-decreasing with DRAM capacity',
    )

# ── Matrix validity ───────────────────────────────────────────────────────────
for policy in POLICIES:
    all_ok &= check(np.all(np.isfinite(energy[policy])),       f'{policy}: finite energy')
    all_ok &= check(np.all(np.isfinite(ttft_latency[policy])), f'{policy}: finite TTFT latency')
    all_ok &= check(np.all(np.isfinite(edp[policy])),          f'{policy}: finite EDP')
    all_ok &= check(np.all(energy[policy]       >= 0),         f'{policy}: non-negative energy')
    all_ok &= check(np.all(ttft_latency[policy] >= 0),         f'{policy}: non-negative TTFT')
    all_ok &= check(np.all(edp[policy]          >= 0),         f'{policy}: non-negative EDP')

# ── No-cache DRAM invariance ──────────────────────────────────────────────────
for j, (name, _reuse) in enumerate(BEIR_DATASETS):
    all_ok &= check(
        np.allclose(energy['no_cache'][:, j],       energy['no_cache'][0, j]),
        f'{name}: no-cache energy invariant to DRAM size',
    )
    all_ok &= check(
        np.allclose(ttft_latency['no_cache'][:, j], ttft_latency['no_cache'][0, j]),
        f'{name}: no-cache TTFT invariant to DRAM size',
    )

print()
print('─' * 60)
print('SANITY CHECK SUMMARY:', 'ALL PASS' if all_ok else 'FAILURES DETECTED — check above')

[PASS] nq: actual reuse within 5% of target
[PASS] nq: trace length = N_QUERIES × K_SIM
[PASS] hotpotqa: actual reuse within 5% of target
[PASS] hotpotqa: trace length = N_QUERIES × K_SIM
[PASS] scidocs: actual reuse within 5% of target
[PASS] scidocs: trace length = N_QUERIES × K_SIM
[PASS] quora: actual reuse within 5% of target
[PASS] quora: trace length = N_QUERIES × K_SIM
[PASS] fever: actual reuse within 5% of target
[PASS] fever: trace length = N_QUERIES × K_SIM
[PASS] fiqa: actual reuse within 5% of target
[PASS] fiqa: trace length = N_QUERIES × K_SIM
[PASS] 1 GB: capacity capped at N_DOCS_TRACE
[PASS] 2 GB: capacity capped at N_DOCS_TRACE
[PASS] 4 GB: capacity capped at N_DOCS_TRACE
[PASS] 8 GB: capacity capped at N_DOCS_TRACE
[PASS] 16 GB: capacity capped at N_DOCS_TRACE
[PASS] 1 GB nq: no-cache hits = 0
[PASS] 1 GB nq: no-cache misses = total
[PASS] 1 GB nq: lfu_l1_l2 total valid
[PASS] 1 GB nq: opt_two_tier total valid
[PASS] 1 GB nq: opt_two_tier <= lfu_l1_l2 <= no-cache
[

## Cell 7 — Generate Plots via Script

In [9]:
import subprocess

print('Legacy plotting cells replaced. Generating standardized Exp3 outputs via script...')
subprocess.run(
    ['python3', '/home/workspace/workspace/scripts/plot_exp3_retrieval_totals.py'],
    check=True,
)
print('Exp3 standardized plots generated.')

"""
# ── Cache-only traffic (per query, in nJ / µs) ─────────────────────────────
# Strips the decoder-dominated non-SIM baseline so policy & DRAM differences
# are visible.  The original SIM doc_scores term is replaced by the cache
# traffic shown here (see Cell 6 for the actual replaced energy / TTFT).
ce_nJ  = {p: cache_energy[p] / 1e3  for p in POLICIES}   # pJ → nJ
ct_us  = {p: cache_ttft[p]   / 1e3  for p in POLICIES}   # ns → µs

# Build (policy × dataset) matrices at the chosen DRAM size.
def _policy_x_dataset(metric_per_policy):
    return np.array([metric_per_policy[p][i_dram, :] for p in POLICIES])

energy_mat  = _policy_x_dataset(ce_nJ)
ttft_mat    = _policy_x_dataset(ct_us)
hitrate_mat = _policy_x_dataset(hitrate)

POLICY_ROW_LABELS = [POLICY_LABELS[p] for p in POLICIES]

def draw_policy_heatmap(ax, data, title, cmap, fmt, vmin=None, vmax=None,
                        text_color_threshold=0.6):
    vmin_eff = vmin if vmin is not None else data.min()
    vmax_eff = vmax if vmax is not None else data.max()
    im = ax.imshow(data, cmap=cmap, aspect='auto', vmin=vmin_eff, vmax=vmax_eff)
    ax.set_xticks(range(ndata))
    ax.set_xticklabels(DATASET_NAMES, rotation=30, ha='right', fontsize=8)
    ax.set_yticks(range(len(POLICIES)))
    ax.set_yticklabels(POLICY_ROW_LABELS, fontsize=8)
    ax.set_title(title, fontsize=10)
    span = max(vmax_eff - vmin_eff, 1e-9)
    threshold = vmin_eff + text_color_threshold * span
    for i in range(len(POLICIES)):
        for j in range(ndata):
            ax.text(j, i, fmt.format(data[i, j]),
                    ha='center', va='center', fontsize=8,
                    color='white' if data[i, j] > threshold else 'black')
    return im

fig, axes = plt.subplots(3, 1, figsize=(9, 10), constrained_layout=True)
fig.suptitle(
    f'Experiment 2: Cache-Only Contribution per Query  @  DRAM = {DRAM_FOCUS_GB} GB\n'
    f'N_DOCS_TRACE={N_DOCS_TRACE:,}, N_QUERIES={N_QUERIES:,}, K_SIM={K_SIM}, '
    f'DRAM={DRAM_BW_GBps:.0f} GB/s, Disk={DISK_BW_GBps:.0f} GB/s\n'
    f'(non-SIM baseline of {nonsim_energy_J*1e3:.1f} mJ / '
    f'{nonsim_latency_s*1e3:.1f} ms per query is added on top)',
    fontsize=10,
)

im_e = draw_policy_heatmap(axes[0], energy_mat,
                           'Cache energy [nJ/query]',
                           'YlOrRd', '{:.2f}', vmin=0)
im_t = draw_policy_heatmap(axes[1], ttft_mat,
                           'Cache TTFT [µs/query]',
                           'YlGnBu', '{:.2f}', vmin=0)
im_h = draw_policy_heatmap(axes[2], hitrate_mat,
                           'Hit rate',
                           'Greens', '{:.2f}', vmin=0, vmax=1.0)

axes[0].set_ylabel('Policy', fontsize=9)
axes[1].set_ylabel('Policy', fontsize=9)
axes[2].set_ylabel('Policy', fontsize=9)
axes[2].set_xlabel('BEIR dataset (increasing reuse ratio →)', fontsize=9)

fig.colorbar(im_e, ax=axes[0], shrink=0.85, pad=0.02,
             label='Cache energy [nJ/query]')
fig.colorbar(im_t, ax=axes[1], shrink=0.85, pad=0.02,
             label='Cache TTFT [µs/query]')
fig.colorbar(im_h, ax=axes[2], shrink=0.85, pad=0.02,
             label='Hit rate')

plt.savefig('/home/workspace/workspace/exp2_heatmaps.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved exp2_heatmaps.png  (DRAM = {DRAM_FOCUS_GB} GB, comparing '
      f'{len(POLICIES)} policies × {ndata} datasets)')

# ── Headline numbers (also printed for the report) ──────────────────────────
def fmt_J(j):
    return f'{j*1e3:>7.2f} mJ' if j >= 1e-3 else f'{j*1e6:>7.2f} µJ' if j >= 1e-6 else f'{j*1e9:>7.2f} nJ'
def fmt_s(s):
    return f'{s*1e3:>7.2f} ms' if s >= 1e-3 else f'{s*1e6:>7.2f} µs' if s >= 1e-6 else f'{s*1e9:>7.2f} ns'

savings_E_pJ  = energy['no_cache'] - energy['lfu']
savings_T_ns  = ttft_latency['no_cache'] - ttft_latency['lfu']
print()
print('=== Headline savings: LFU vs No-Cache (mean across datasets) ===')
for i, gb in enumerate(DRAM_GB_VALUES):
    e_save_J = savings_E_pJ[i].mean() * 1e-12
    t_save_s = savings_T_ns[i].mean() * 1e-9
    print(f'  DRAM {gb:>5g} GB:  ΔE = {fmt_J(e_save_J)},  ΔTTFT = {fmt_s(t_save_s)}')

print()
print('=== Per-dataset LFU hit rate at smallest DRAM ===')
for j, (name, reuse) in enumerate(BEIR_DATASETS):
    print(f'  {name:<10} reuse={reuse:.2f}  '
          f'LFU={hitrate["lfu"][0, j]:.3f}  OPT={hitrate["opt"][0, j]:.3f}  '
          f'gap={hitrate["opt"][0, j] - hitrate["lfu"][0, j]:.3f}')

# ── Stage-by-stage retrieval comparison (Option A: include AccelForge baseline) ──
# Compares the four retrieval strategies on a log scale.  Decoder cost is excluded
# so this is an apples-to-apples retrieval-only comparison; the corresponding
# full-pipeline numbers are obtained by adding the non-SIM baseline
# (≈ 5.52 J / 687.6 ms) to every bar.
af_e_nJ = sim_energy_J  * 1e9   # AccelForge brute-force doc_scores compute
af_t_us = sim_latency_s * 1e6

def _mean_e(p):
    return cache_energy[p].mean() / 1e3   # pJ → nJ
def _mean_t(p):
    return cache_ttft[p].mean()  / 1e3    # ns → µs

stage_names = [
    'AccelForge\n(brute-force\ndoc_scores)',
    'No Cache\n(indexed,\nall miss)',
    'LFU\n(EdgeRAG, mean)',
    'Bélády OPT\n(mean)',
]
stage_e_nJ  = [af_e_nJ, _mean_e('no_cache'), _mean_e('lfu'), _mean_e('opt')]
stage_t_us  = [af_t_us, _mean_t('no_cache'), _mean_t('lfu'), _mean_t('opt')]
stage_color = ['#cc4c4c', '#dd9c4c', '#5a8cc8', '#5fc080']

def _label_e(v):
    if v >= 1e6:  return f'{v/1e6:.1f} mJ'
    if v >= 1e3:  return f'{v/1e3:.1f} µJ'
    return f'{v:.1f} nJ'

def _label_t(v):
    if v >= 1e6:  return f'{v/1e6:.2f} s'
    if v >= 1e3:  return f'{v/1e3:.1f} ms'
    return f'{v:.2f} µs'

fig, axes = plt.subplots(1, 2, figsize=(11, 4.4), constrained_layout=True)
fig.suptitle(
    'Retrieval cost per query: AccelForge brute-force vs cache-modelled indexed retrieval\n'
    '(decoder cost excluded; apples-to-apples retrieval-only; linear scale)',
    fontsize=10,
)

for ax, vals, ylabel, fmtter, title in [
    (axes[0], stage_e_nJ, 'Retrieval energy [nJ/query]', _label_e, 'Energy'),
    (axes[1], stage_t_us, 'Retrieval TTFT [µs/query]',   _label_t, 'Time-to-first-token'),
]:
    ax.bar(stage_names, vals, color=stage_color, edgecolor='white')
    ax.set_ylabel(ylabel, fontsize=9)
    ax.set_title(title, fontsize=9)
    for i, v in enumerate(vals):
        ax.text(i, v * 1.025, fmtter(v), ha='center', va='bottom', fontsize=8)
    ax.tick_params(axis='x', labelsize=7.5)
    ax.set_ylim(top=max(vals) * 1.18)

plt.savefig('/home/workspace/workspace/exp2_baseline_compare.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved exp2_baseline_compare.png')

# ── Numerical stage table (with AccelForge multipliers) ──
flat_names = [s.replace('\n', ' ') for s in stage_names]
nw = max(len(n) for n in flat_names)
print()
print('=== Stage-by-stage retrieval cost (per query) ===')
for n, e, t in zip(flat_names, stage_e_nJ, stage_t_us):
    print(f'  {n:<{nw}}   E={_label_e(e):>10}   TTFT={_label_t(t):>10}')

print()
print('=== Multipliers vs AccelForge brute-force (energy / TTFT divided by) ===')
ref_e, ref_t = stage_e_nJ[0], stage_t_us[0]
for n, e, t in zip(flat_names, stage_e_nJ, stage_t_us):
    print(f'  {n:<{nw}}   E ÷ {ref_e/e:>11,.0f}×   TTFT ÷ {ref_t/t:>11,.0f}×')

# ── Retrieval-cost Pareto: every (DRAM, dataset, policy) operating point + AF baseline ──
# Retrieval-only (decoder excluded) so the AccelForge brute-force baseline and the
# cache operating points are visually separated; on a full-pipeline view they would
# overlap at log(5.5 J) / log(688 s) and the Pareto would be uninformative.
from matplotlib.lines import Line2D

fig, ax = plt.subplots(1, 1, figsize=(9, 6.5), constrained_layout=True)

policy_colors  = {'no_cache': '#dd9c4c', 'lfu': '#5a8cc8', 'opt': '#5fc080'}
policy_markers = {'no_cache': 'o',       'lfu': 's',       'opt': '^'}
# marker size grows with DRAM (0.128 → 4 GB)
size_scale     = {0.128: 25, 0.25: 35, 0.5: 60, 1: 90, 2: 130, 4: 180}

for p in POLICIES:
    for i, gb in enumerate(DRAM_GB_VALUES):
        ce_J = cache_energy[p][i, :] * 1e-12   # pJ → J
        ct_s = cache_ttft[p][i, :]   * 1e-9    # ns → s
        ax.scatter(ct_s, ce_J,
                   marker=policy_markers[p], s=size_scale[gb],
                   c=policy_colors[p], edgecolors='white',
                   alpha=0.65, linewidth=0.6)

# AccelForge brute-force baseline (single star at the upper-right)
ax.scatter(sim_latency_s, sim_energy_J, marker='*', s=600,
           c='#cc4c4c', edgecolors='black', linewidth=1.2, zorder=10)
ax.annotate(
    f'AccelForge brute-force\n{sim_energy_J*1e3:.1f} mJ / {sim_latency_s:.2f} s',
    xy=(sim_latency_s, sim_energy_J),
    xytext=(-110, -55), textcoords='offset points',
    fontsize=9, ha='left',
    arrowprops=dict(arrowstyle='->', color='black', lw=0.7),
)

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Retrieval TTFT [s, log]', fontsize=10)
ax.set_ylabel('Retrieval energy [J, log]', fontsize=10)
ax.set_title(
    'Retrieval-cost Pareto: AccelForge brute-force vs cached indexed retrieval\n'
    f'every (DRAM × dataset × policy) operating point — '
    f'{len(POLICIES)*len(DRAM_GB_VALUES)*len(BEIR_DATASETS)} cache points + 1 baseline',
    fontsize=10,
)
ax.grid(True, which='both', alpha=0.3, linestyle=':')

legend_handles = [
    Line2D([0], [0], marker='*', color='w', markerfacecolor='#cc4c4c',
           markeredgecolor='black', markersize=15,
           label='AccelForge brute-force doc_scores'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#dd9c4c',
           markeredgecolor='white', markersize=9, label='No Cache (indexed)'),
    Line2D([0], [0], marker='s', color='w', markerfacecolor='#5a8cc8',
           markeredgecolor='white', markersize=9, label='LFU (EdgeRAG)'),
    Line2D([0], [0], marker='^', color='w', markerfacecolor='#5fc080',
           markeredgecolor='white', markersize=9, label='Bélády OPT'),
]
ax.legend(handles=legend_handles, loc='center right', fontsize=9, frameon=True)
ax.text(0.02, 0.02, 'marker size ∝ DRAM capacity (0.25 → 4 GB)',
        transform=ax.transAxes, fontsize=8, va='bottom', ha='left',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                  edgecolor='gray', alpha=0.85))

plt.savefig('/home/workspace/workspace/exp2_pareto.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved exp2_pareto.png')

# ── Zoomed-in view of the cache cluster (AccelForge baseline off-chart) ──────
# Single panel, linear axes in nJ / µs so the LFU↔OPT spread is readable.
# Encoding matches the original Pareto (shape + color = policy, size = DRAM);
# a thin line connects each (policy, dataset) DRAM sweep so the saturation
# pattern is visible.  AccelForge baseline omitted — it sits ~6 orders of
# magnitude up-and-right and would collapse the cluster to a point if shown.
size_scale_zoom = {0.128: 38, 0.25: 55, 0.5: 95, 1: 150, 2: 215, 4: 295}

fig, ax = plt.subplots(1, 1, figsize=(9.5, 6.5), constrained_layout=True)

for p in POLICIES:
    for j, (name, _reuse) in enumerate(BEIR_DATASETS):
        ce_nJ = cache_energy[p][:, j] / 1e3   # pJ → nJ
        ct_us = cache_ttft[p][:, j]   / 1e3   # ns → µs
        ax.plot(ct_us, ce_nJ, '-', color=policy_colors[p],
                alpha=0.25, lw=0.8, zorder=1)
        for i, gb in enumerate(DRAM_GB_VALUES):
            ax.scatter(ct_us[i], ce_nJ[i],
                       marker=policy_markers[p], s=size_scale_zoom[gb],
                       facecolor=policy_colors[p], edgecolor='white',
                       linewidth=0.7, alpha=0.85, zorder=3)
        if p == 'lfu':
            ax.annotate(name, xy=(ct_us[0], ce_nJ[0]),
                        xytext=(6, 4), textcoords='offset points',
                        fontsize=7.5, color='#333', fontweight='bold')

ax.set_xlabel('Cache TTFT [µs / query]', fontsize=10)
ax.set_ylabel('Cache energy [nJ / query]', fontsize=10)
ax.set_title(
    f'Cache-cluster zoom: {len(POLICIES)*len(DRAM_GB_VALUES)*len(BEIR_DATASETS)} '
    '(DRAM × dataset × policy) operating points\n'
    f'AccelForge baseline at ({sim_latency_s:.2f} s, {sim_energy_J*1e3:.1f} mJ) is off-chart',
    fontsize=10,
)
ax.grid(True, alpha=0.3, linestyle=':')

policy_handles = [
    Line2D([0], [0], marker=policy_markers[p], color='w',
           markerfacecolor=policy_colors[p], markeredgecolor='white',
           markersize=10, label=POLICY_LABELS[p])
    for p in POLICIES
]
size_handles = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#888',
           markeredgecolor='white',
           markersize=np.sqrt(size_scale_zoom[gb]),
           label=f'{gb} GB DRAM')
    for gb in DRAM_GB_VALUES
]
leg1 = ax.legend(handles=policy_handles, loc='upper left', fontsize=9,
                 title='policy', title_fontsize=9, frameon=True)
ax.add_artist(leg1)
ax.legend(handles=size_handles, loc='lower left', fontsize=8,
          title='marker size', title_fontsize=8,
          labelspacing=1.1, borderpad=0.7, frameon=True)

plt.savefig('/home/workspace/workspace/exp2_pareto_zoom.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved exp2_pareto_zoom.png')
"""

Legacy plotting cells replaced. Generating standardized Exp3 outputs via script...


Wrote /home/workspace/workspace/exp3_policy_compare.csv
Wrote /home/workspace/workspace/exp3_baseline_compare_log.png
Wrote /home/workspace/workspace/exp3_reuse_compare.png
Wrote /home/workspace/workspace/exp3_average_totals.png
Wrote /home/workspace/workspace/exp3_hit_breakdown.png


Exp3 standardized plots generated.


'\n# ── Cache-only traffic (per query, in nJ / µs) ─────────────────────────────\n# Strips the decoder-dominated non-SIM baseline so policy & DRAM differences\n# are visible.  The original SIM doc_scores term is replaced by the cache\n# traffic shown here (see Cell 6 for the actual replaced energy / TTFT).\nce_nJ  = {p: cache_energy[p] / 1e3  for p in POLICIES}   # pJ → nJ\nct_us  = {p: cache_ttft[p]   / 1e3  for p in POLICIES}   # ns → µs\n\n# Build (policy × dataset) matrices at the chosen DRAM size.\ndef _policy_x_dataset(metric_per_policy):\n    return np.array([metric_per_policy[p][i_dram, :] for p in POLICIES])\n\nenergy_mat  = _policy_x_dataset(ce_nJ)\nttft_mat    = _policy_x_dataset(ct_us)\nhitrate_mat = _policy_x_dataset(hitrate)\n\nPOLICY_ROW_LABELS = [POLICY_LABELS[p] for p in POLICIES]\n\ndef draw_policy_heatmap(ax, data, title, cmap, fmt, vmin=None, vmax=None,\n                        text_color_threshold=0.6):\n    vmin_eff = vmin if vmin is not None else data.min()\n    

## Cell 8 — Legacy plot cells disabled

In [10]:
print('Legacy secondary plot cell disabled; use script outputs (exp3_reuse_compare / exp3_hit_breakdown).')

"""
MAINMEM_FOCUS_GB = 8
MAINMEM_FOCUS_IDX = DRAM_GB_VALUES.index(MAINMEM_FOCUS_GB)

ce_nJ  = {p: cache_energy[p] / 1e3 for p in POLICIES}   # pJ -> nJ
ct_us  = {p: cache_ttft[p]   / 1e3 for p in POLICIES}   # ns -> us

policy_colors  = {'no_cache': '#e07070', 'lfu': '#70aae0', 'opt': '#70c87a'}
policy_markers = {'no_cache': 'o',       'lfu': 's',       'opt': '^'}
x = np.arange(ndata)
x_labels = [f'{name}\n{reuse:.2f}x' for name, reuse in BEIR_DATASETS]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.4), constrained_layout=True)
fig.suptitle(
    'Experiment 2: No Cache vs LFU vs Belady OPT @ fixed MainMemory (DRAM)\n'
    f'GlobalBuffer={GLOBAL_BUFFER_MB} MB, MainMemory={MAINMEM_FOCUS_GB} GB, K_SIM={K_SIM}',
    fontsize=10,
)

panels = [
    (axes[0], ce_nJ, 'Cache energy [nJ/query]', 'Cache energy'),
    (axes[1], ct_us, 'Cache TTFT [us/query]', 'Time-to-first-token contribution'),
    (axes[2], hitrate, 'Hit rate', 'Cache hit rate'),
]

for ax, data, ylabel, title in panels:
    for p in POLICIES:
        ax.plot(x, data[p][MAINMEM_FOCUS_IDX, :],
                marker=policy_markers[p],
                color=policy_colors[p],
                label=POLICY_LABELS[p],
                linewidth=1.7, markersize=5)
    ax.set_xticks(x)
    ax.set_xticklabels(x_labels, rotation=20, ha='right', fontsize=8)
    ax.set_ylabel(ylabel, fontsize=9)
    ax.set_title(title, fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)

axes[1].set_xlabel('BEIR dataset (reuse ratio shown under label)', fontsize=9)
axes[2].set_ylim(-0.02, 1.05)

plt.savefig('/home/workspace/workspace/exp2_gap_hitrate.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved exp2_gap_hitrate.png  (fixed MainMemory = {MAINMEM_FOCUS_GB} GB, x=BEIR reuse ratios)')
"""

Legacy secondary plot cell disabled; use script outputs (exp3_reuse_compare / exp3_hit_breakdown).


"\nMAINMEM_FOCUS_GB = 8\nMAINMEM_FOCUS_IDX = DRAM_GB_VALUES.index(MAINMEM_FOCUS_GB)\n\nce_nJ  = {p: cache_energy[p] / 1e3 for p in POLICIES}   # pJ -> nJ\nct_us  = {p: cache_ttft[p]   / 1e3 for p in POLICIES}   # ns -> us\n\npolicy_colors  = {'no_cache': '#e07070', 'lfu': '#70aae0', 'opt': '#70c87a'}\npolicy_markers = {'no_cache': 'o',       'lfu': 's',       'opt': '^'}\nx = np.arange(ndata)\nx_labels = [f'{name}\n{reuse:.2f}x' for name, reuse in BEIR_DATASETS]\n\nfig, axes = plt.subplots(1, 3, figsize=(15, 4.4), constrained_layout=True)\nfig.suptitle(\n    'Experiment 2: No Cache vs LFU vs Belady OPT @ fixed MainMemory (DRAM)\n'\n    f'GlobalBuffer={GLOBAL_BUFFER_MB} MB, MainMemory={MAINMEM_FOCUS_GB} GB, K_SIM={K_SIM}',\n    fontsize=10,\n)\n\npanels = [\n    (axes[0], ce_nJ, 'Cache energy [nJ/query]', 'Cache energy'),\n    (axes[1], ct_us, 'Cache TTFT [us/query]', 'Time-to-first-token contribution'),\n    (axes[2], hitrate, 'Hit rate', 'Cache hit rate'),\n]\n\nfor ax, data, ylabel, 

## Cell 9 — Legacy table cell disabled

In [11]:
print('Legacy raw-results cell disabled; canonical table is written by plot_exp3_retrieval_totals.py')

"""
rows = []
for i, gb in enumerate(DRAM_GB_VALUES):
    capacity = embedding_capacity(gb, N_DOCS_TRACE)
    for j, (name, reuse) in enumerate(BEIR_DATASETS):
        for p in POLICIES:
            res = sim_results[gb][name][p]
            tc = tier_counts[gb][name][p]
            rows.append({
                'MAIN_MEMORY_GB':      gb,
                'GLOBAL_BUFFER_MB':    GLOBAL_BUFFER_MB,
                'capacity_embeddings': capacity,
                'dataset':             name,
                'reuse':               reuse,
                'policy':              p,
                'h2':                  tc['h2'],
                'h_dram':              tc['h_dram'],
                'h_disk':              tc['h_disk'],
                'hits_total':          res.hits,
                'misses':              res.misses,
                'hit_rate':            f'{res.hit_rate:.3f}',
                'energy_nJ_per_query': f'{energy[p][i,j]/1e3:.1f}',
                'ttft_us_per_query':   f'{ttft_latency[p][i,j]/1e3:.2f}',
                'edp_pJ_ns':           f'{edp[p][i,j]:.2e}',
            })

df = pd.DataFrame(rows)
pd.set_option('display.max_rows', 120)
display(df)
df.to_csv('/home/workspace/workspace/exp2_raw_results.csv', index=False)
print(f'Saved exp2_raw_results.csv  ({len(df)} rows)')
"""

Legacy raw-results cell disabled; canonical table is written by plot_exp3_retrieval_totals.py


"\nrows = []\nfor i, gb in enumerate(DRAM_GB_VALUES):\n    capacity = embedding_capacity(gb, N_DOCS_TRACE)\n    for j, (name, reuse) in enumerate(BEIR_DATASETS):\n        for p in POLICIES:\n            res = sim_results[gb][name][p]\n            tc = tier_counts[gb][name][p]\n            rows.append({\n                'MAIN_MEMORY_GB':      gb,\n                'GLOBAL_BUFFER_MB':    GLOBAL_BUFFER_MB,\n                'capacity_embeddings': capacity,\n                'dataset':             name,\n                'reuse':               reuse,\n                'policy':              p,\n                'h2':                  tc['h2'],\n                'h_dram':              tc['h_dram'],\n                'h_disk':              tc['h_disk'],\n                'hits_total':          res.hits,\n                'misses':              res.misses,\n                'hit_rate':            f'{res.hit_rate:.3f}',\n                'energy_nJ_per_query': f'{energy[p][i,j]/1e3:.1f}',\n               